### Box Office Bomb Data Pipeline Assignment

**Total Marks: 20**

This notebook implements an end-to-end pipeline to:
1. Scrape box office bomb data from Wikipedia
2. Validate & clean data using Pydantic
3. Enrich with OMDb API data
4. Perform consistency checks
5. Create final categorized dataset

## Setup and Imports

In [ ]:
# Import required libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, field_validator, ValidationError
from typing import Optional
import re
import time
from pathlib import Path

## Task 1: Scrape the "Bombs" Table (4 Marks)

Extract raw data from the Wikipedia HTML file:
- Film Title (with symbols like § and †)
- Year
- Net production budget (may contain ranges like "$100–160")
- Estimated loss (Nominal column)

Note:
- You need to extract the entire raw string with the symbols, references, etc along with the titles.
- You must handle the nested headers in the Wikipedia table (Budget and Loss columns have sub-headers).

In [ ]:
def scrape_box_office_bombs():
    """
    Scrape the box office bombs table from the local HTML file.
    Returns a list of dictionaries with raw extracted data.
    """
    # TODO: Load the HTML file
    url = "https://en.wikipedia.org/wiki/List_of_biggest_box-office_bombs"
    headers = {
        "User-Agent": "BoxOfficeBombsScraper/1.0"
    }

    response = requests.get(url, headers=headers)
    html = response.text

    # TODO: Parse with BeautifulSoup
    soup = BeautifulSoup(html, 'html.parser')


    # TODO: Find the main table - it's wikitable sortable with caption "Biggest box-office bombs"
    table = soup.find('table', {'class': 'wikitable'})
    if table is None:
        raise ValueError("Table not found")

    raw_data = []

    # TODO: Iterate through rows (skip headers) and extract: 'raw_title', 'raw_year', 'raw_budget', 'raw_loss'.
    # Store extracted data in raw_data. Example entry: {'raw_title': 'Town & Country', 'raw_year': '2001', 'raw_budget': '$90', 'raw_loss': '$10.4'}
    for row in table.find_all('tr')[1:]:
      title_cell = row.find('th')
      cells = row.find_all('td')

      if title_cell and len(cells) >= 4:
          raw_title = title_cell.get_text().strip()
          raw_year = cells[0].get_text().strip()
          raw_budget = cells[1].get_text().strip()
          raw_loss = cells[3].get_text().strip()

          raw_data.append({
              'raw_title': raw_title,
              'raw_year': raw_year,
              'raw_budget': raw_budget,
              'raw_loss': raw_loss
          })


    return raw_data

In [ ]:
# Test the scraping function
raw_movies = scrape_box_office_bombs()
raw_movies
print(f"Scraped {len(raw_movies)} movies")
print("\n first 5 raw entries:")
for i, movie in enumerate(raw_movies[-5:]):
    print(f"\n{i+1}. {movie}")

print("\n last 5 raw entries:")

for i, movie in enumerate(raw_movies[-5:]):
    print(f"\n{i+1}. {movie}")

Scraped 139 movies

 first 5 raw entries:

1. {'raw_title': 'The Wolfman', 'raw_year': '2010', 'raw_budget': '$150', 'raw_loss': '$76'}

2. {'raw_title': 'Wonder Woman 1984 §', 'raw_year': '2020', 'raw_budget': '$200', 'raw_loss': '$100–135'}

3. {'raw_title': 'A Wrinkle in Time', 'raw_year': '2018', 'raw_budget': '$125', 'raw_loss': '$130.6'}

4. {'raw_title': 'xXx: State of the Union', 'raw_year': '2005', 'raw_budget': '$113.1', 'raw_loss': '$78'}

5. {'raw_title': 'Zoom', 'raw_year': '2006', 'raw_budget': '$75.6', 'raw_loss': '$69'}

 last 5 raw entries:

1. {'raw_title': 'The Wolfman', 'raw_year': '2010', 'raw_budget': '$150', 'raw_loss': '$76'}

2. {'raw_title': 'Wonder Woman 1984 §', 'raw_year': '2020', 'raw_budget': '$200', 'raw_loss': '$100–135'}

3. {'raw_title': 'A Wrinkle in Time', 'raw_year': '2018', 'raw_budget': '$125', 'raw_loss': '$130.6'}

4. {'raw_title': 'xXx: State of the Union', 'raw_year': '2005', 'raw_budget': '$113.1', 'raw_loss': '$78'}

5. {'raw_title': 'Zoom'

## Task 2: Pydantic Data Parsing & Validation (6 Marks)

Create a Pydantic model that:
- Cleans titles (removes §, †, and footnotes like [nb 2], [1])
- Parses numeric values (handles ranges, currency symbols)
- Validates year as integer

In [ ]:
class MovieData(BaseModel):
    """
    Pydantic model for validating and cleaning movie data.
    """
    # TODO: Define the 4 required fields with their types:
    # title (str), year (int), budget_millions (float), loss_millions (float)
    title: str
    year: int
    budget_millions: float
    loss_millions: float

    @field_validator('title', mode='before')
    @classmethod
    def clean_title(cls, v):
        """
        Remove footnote markers and special symbols from title.
        - Remove § (streaming symbol)
        - Remove † (currently playing symbol)
        - Remove footnotes like [nb 2], [1], etc.
        """
        # TODO: Implement title cleaning logic
        # Hint: Use .replace() for symbols and re.sub() for footnotes
        v = v.replace('§', '').replace('†', '')
        v = re.sub(r'\[\w+ \d+\]', '', v)

        return v.strip()

    @field_validator('year', mode='before')
    @classmethod
    def validate_year(cls, v):
        """
        Ensure year is a valid integer.
        Remove any extra characters and convert to int.
        """
        # TODO: Clean string (remove non-digits) and convert to integer
        v = int(re.sub(r'\D', '', v))
        return v

    @field_validator('budget_millions', mode='before')
    @classmethod
    def parse_budget(cls, v):
        """
        Parse budget value:
        - Strip $ and other currency symbols
        - Handle ranges (e.g., "100–160") by calculating average
        - Remove reference tags
        """
        # TODO: Call your helper method _parse_numeric_value
        return cls._parse_numeric_value(v)


    @field_validator('loss_millions', mode='before')
    @classmethod
    def parse_loss(cls, v):
        """
        Parse loss value with same logic as budget.
        """
        # TODO: Call your helper method _parse_numeric_value
        return cls._parse_numeric_value(v)

    @staticmethod
    def _parse_numeric_value(v):
        """
        Helper method to parse numeric values with ranges.
        """
        # TODO: Implement logic to:
        # 1. Clean string (remove $, reference tags)
        # 2. Check for ranges (hyphen or en-dash) and calculate average
        # 3. Return a float
        v = v.replace('$', '').replace('–', '-').replace('—', '-')
        v = re.sub(r'\[\w+ \d+\]', '', v)
        if '-' in v:
            v = (float(v.split('-')[0]) + float(v.split('-')[1])) / 2
        else:
            v = float(v)
        return v

In [ ]:
# Validate and clean the raw data
validated_movies = []
failed_validations = []

for raw_movie in raw_movies:
    try:
        movie = MovieData(
            title=raw_movie['raw_title'],
            year=raw_movie['raw_year'],
            budget_millions=raw_movie['raw_budget'],
            loss_millions=raw_movie['raw_loss']
        )
        validated_movies.append(movie)
    except ValidationError as e:
        failed_validations.append({
            'raw_data': raw_movie,
            'error': str(e)
        })
        print(f"Validation failed for {raw_movie['raw_title']}: {e}")

print(f"\n{'='*60}")
print(f"Validation Results:")
print(f"Successfully validated: {len(validated_movies)} movies")
print(f"Failed validations: {len(failed_validations)} movies")
print(f"{'='*60}")

# Show first 3 validated movies
print("\nLast 15 validated movies:")
for i, movie in enumerate(validated_movies[-15:]):
    print(f"\n{i+1}. {movie.model_dump()}")

Validation failed for Transformers: The Last Knight: 1 validation error for MovieData
loss_millions
  Value error, could not convert string to float: '100+' [type=value_error, input_value='$100+', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Validation Results:
Successfully validated: 138 movies
Failed validations: 1 movies

Last 15 validated movies:

1. {'title': 'Tomorrowland', 'year': 2015, 'budget_millions': 185.0, 'loss_millions': 120.0}

2. {'title': 'Town & Country', 'year': 2001, 'budget_millions': 90.0, 'loss_millions': 85.0}

3. {'title': 'Treasure Planet', 'year': 2002, 'budget_millions': 140.0, 'loss_millions': 85.0}

4. {'title': 'Tron: Ares', 'year': 2025, 'budget_millions': 220.0, 'loss_millions': 132.7}

5. {'title': 'Turning Red', 'year': 2022, 'budget_millions': 175.0, 'loss_millions': 173.0}

6. {'title': 'Valerian and the City of a Thousand Planets', 'year': 2017, 'budget_millions': 178.6, 'loss_millions': 82.0}



## Task 3: Enrich with OMDb Data (4 Marks)

Query OMDb API for each movie to get:
- Plot
- Metascore
- IMDb Rating
- Director
- Language

Handle API failures (Response='False') or missing fields ('N/A') gracefully by storing them as None. Do not delete the row.

In [ ]:
# OMDb API configuration
# NOTE: You need to get a free API key from http://www.omdbapi.com/apikey.aspx
OMDB_API_KEY = "f9735dc3"
OMDB_BASE_URL = "http://www.omdbapi.com/"

def query_omdb(title: str, year: int) -> dict:
    """
    Query OMDb API for movie metadata.
    Returns dict with plot, metascore, imdb_rating, director, language, omdb_year.
    """
    params = {
        'apikey': OMDB_API_KEY,
        't': title,
        'y': year
    }

    try:
        response = requests.get(OMDB_BASE_URL, params=params, timeout=10)
        data = response.json()
    except Exception:
        return {
            'Plot': None,
            'Metascore': None,
            'IMDb_Rating': None,
            'Director': None,
            'Language': None,
            'OMDb_Year': None
        }

    if data.get('Response') == 'False':
        return {
            'Plot': None,
            'Metascore': None,
            'IMDb_Rating': None,
            'Director': None,
            'Language': None,
            'OMDb_Year': None
        }

    def clean_value(val, numeric=False):
        if val == 'N/A' or val is None:
            return None
        return float(val) if numeric else val

    return {
        'Plot': clean_value(data.get('Plot')),
        'Metascore': clean_value(data.get('Metascore'), numeric=True),
        'IMDb_Rating': clean_value(data.get('imdbRating'), numeric=True),
        'Director': clean_value(data.get('Director')),
        'Language': clean_value(data.get('Language')),
        'OMDb_Year': clean_value(data.get('Year'))
    }


In [ ]:
enriched_data = []

print("Querying OMDb API...")
i=0
for movie in validated_movies:
    omdb_data = query_omdb(movie.title, movie.year)

    enriched_data.append({
        'Title': movie.title,
        'Year': movie.year,
        'Budget_Millions': movie.budget_millions,
        'Loss_Millions': movie.loss_millions,
        'Director': omdb_data['Director'],
        'Language': omdb_data['Language'],
        'IMDb_Rating': omdb_data['IMDb_Rating'],
        'Metascore': omdb_data['Metascore'],
        'Plot': omdb_data['Plot'],
        'OMDb_Year': omdb_data['OMDb_Year']
    })

        # Print progress every 5 movies
    if (i + 1) % 5 == 0 or i == 0:
        print(f"  Processed {i+1}/{len(validated_movies)}: {movie.title}")
    i+=1

    time.sleep(0.2)  # polite API usage

print(f"Enriched {len(enriched_data)} movies")


Querying OMDb API...
  Processed 1/138: The 13th Warrior
  Processed 5/138: The Adventures of Pluto Nash
  Processed 10/138: Allied
  Processed 15/138: Battlefield Earth
  Processed 20/138: Black Adam
  Processed 25/138: Chaos Walking
  Processed 30/138: Cutthroat Island
  Processed 35/138: Dolittle
  Processed 40/138: Fantastic Four
  Processed 45/138: Furiosa: A Mad Max Saga
  Processed 50/138: The Good Dinosaur
  Processed 55/138: Haunted Mansion
  Processed 60/138: Hugo
  Processed 65/138: Ishtar
  Processed 70/138: Jungle Cruise
  Processed 75/138: The Last Duel
  Processed 80/138: Mars Needs Moms
  Processed 85/138: Monster Trucks
  Processed 90/138: The New Mutants
  Processed 95/138: Peter Pan
  Processed 100/138: Red Planet
  Processed 105/138: Sinbad: Legend of the Seven Seas
  Processed 110/138: Son of the Mask
  Processed 115/138: Sphere
  Processed 120/138: Tenet
  Processed 125/138: Town & Country
  Processed 130/138: West Side Story
  Processed 135/138: Wonder Woman 1984

In [ ]:
# Print enrichment statistics
print("\n" + "="*60)
print("OMDb ENRICHMENT RESULTS")
print("="*60)

# Count missing data
missing_ratings = sum(1 for item in enriched_data if item['IMDb_Rating'] is None)
missing_directors = sum(1 for item in enriched_data if item['Director'] is None)
missing_plots = sum(1 for item in enriched_data if item['Plot'] is None)

print(f"Total enriched movies: {len(enriched_data)}")
print(f"Movies missing IMDb rating: {missing_ratings}")
print(f"Movies missing Director: {missing_directors}")
print(f"Movies missing Plot: {missing_plots}")

# Show sample enriched data
if enriched_data:
    print("\nSAMPLE: Enriched Data")
    print("-" * 80)
    for i in range(min(3, len(enriched_data))):
        item = enriched_data[i]
        print(f"\nMovie {i+1}: {item['Title']} ({item['Year']})")
        print(f"  Director: {item['Director'] or 'N/A'}")
        print(f"  Language: {item['Language'] or 'N/A'}")
        print(f"  IMDb: {item['IMDb_Rating'] or 'N/A'}")
        print(f"  Metascore: {item['Metascore'] or 'N/A'}")
        print(f"  Budget: ${item['Budget_Millions']}M")
        print(f"  Loss: ${item['Loss_Millions']}M")


OMDb ENRICHMENT RESULTS
Total enriched movies: 138
Movies missing IMDb rating: 2
Movies missing Director: 2
Movies missing Plot: 2

SAMPLE: Enriched Data
--------------------------------------------------------------------------------

Movie 1: The 13th Warrior (1999)
  Director: John McTiernan
  Language: English, Latin, Swedish, Norse, Old, Danish, Arabic
  IMDb: 6.6
  Metascore: 42.0
  Budget: $130.0M
  Loss: $99.0M

Movie 2: The 355 (2022)
  Director: Simon Kinberg
  Language: English, Chinese, Spanish, French, German, Arabic, Romanian, Hindi, Sign 
  IMDb: 5.6
  Metascore: 40.0
  Budget: $57.5M
  Loss: $93.0M

Movie 3: 47 Ronin (2013)
  Director: Carl Rinsch
  Language: English, Japanese
  IMDb: 6.2
  Metascore: 28.0
  Budget: $200.0M
  Loss: $96.0M


## Task 4: Data Consistency Check (2 Marks)

Compare Wikipedia year with OMDb year:
- "Verified": Years match (±1 year tolerance)
- "Mismatch": Years differ by >1
- "Not Found": OMDb returned no data

In [ ]:
def year_match_status(wiki_year: int, omdb_year: Optional[str]) -> str:
    if omdb_year is None:
        return "Not Found"

    try:
        omdb_year_int = int(re.findall(r'\d{4}', omdb_year)[0])
    except Exception:
        return "Not Found"

    if abs(wiki_year - omdb_year_int) <= 1:
        return "Verified"
    else:
        return "Mismatch"


In [ ]:
for item in enriched_data:
    item['Match_Status'] = year_match_status(
        item['Year'],
        item['OMDb_Year']
    )


In [ ]:
# Print match status results
match_counts = {}
for item in enriched_data:
    status = item['Match_Status']
    match_counts[status] = match_counts.get(status, 0) + 1

print("\n" + "="*60)
print("YEAR MATCH STATUS")
print("="*60)
for status, count in match_counts.items():
    print(f"{status}: {count} movies")


YEAR MATCH STATUS
Verified: 136 movies
Not Found: 2 movies


## Task 5: Final Dataset & Categorization (4 Marks)

Create final DataFrame with:
- Loss_Category based on estimated loss:
  - "Catastrophic": Loss ≥ \$100M

  - "Severe": Loss between \$50M and \$100M

  - "Moderate": Loss < \$50M
- Save to `box_office_failures.csv`

Required columns: Title, Year, Director, Language, Budget_Millions, Loss_Millions, Loss_Category, IMDb_Rating, Metascore, Match_Status

In [ ]:
df = pd.DataFrame(enriched_data)
def categorize_loss(loss):
    if loss >= 100:
        return "Catastrophic"
    elif 50 <= loss < 100:
        return "Severe"
    else:
        return "Moderate"

df['Loss_Category'] = df['Loss_Millions'].apply(categorize_loss)
final_df = df[[
    'Title',
    'Year',
    'Director',
    'Language',
    'Budget_Millions',
    'Loss_Millions',
    'Loss_Category',
    'IMDb_Rating',
    'Metascore',
    'Match_Status'
]]

output_path = Path("box_office_failures.csv")
final_df.to_csv(output_path, index=False)

print(f"Saved final dataset to {output_path.resolve()}")


Saved final dataset to /content/box_office_failures.csv


In [ ]:
# Print final dataset characteristics
print("\n" + "="*60)
print("FINAL DATASET CHARACTERISTICS")
print("="*60)

print(f"Total movies in final dataset: {len(final_df)}")
print(f"Dataset shape: {final_df.shape}")

print(f"\nCOLUMNS:")
for col in final_df.columns:
    non_null = final_df[col].notna().sum()
    null_percent = (final_df[col].isna().sum() / len(final_df)) * 100
    print(f"  {col}: {non_null} non-null ({null_percent:.1f}% null)")

print(f"\nLOSS CATEGORY DISTRIBUTION:")
loss_counts = final_df['Loss_Category'].value_counts()
for category, count in loss_counts.items():
    percentage = (count / len(final_df)) * 100
    print(f"  {category}: {count} movies ({percentage:.1f}%)")

print(f"\nMATCH STATUS DISTRIBUTION:")
match_counts = final_df['Match_Status'].value_counts()
for status, count in match_counts.items():
    percentage = (count / len(final_df)) * 100
    print(f"  {status}: {count} movies ({percentage:.1f}%)")

print(f"\nRATING STATISTICS:")
if 'IMDb_Rating' in final_df.columns and final_df['IMDb_Rating'].notna().any():
    print(f"  IMDb Rating - Mean: {final_df['IMDb_Rating'].mean():.2f}")
    print(f"  IMDb Rating - Min: {final_df['IMDb_Rating'].min():.2f}")
    print(f"  IMDb Rating - Max: {final_df['IMDb_Rating'].max():.2f}")

print(f"\nFINANCIAL STATISTICS:")
print(f"  Average Budget: ${final_df['Budget_Millions'].mean():.1f}M")
print(f"  Average Loss: ${final_df['Loss_Millions'].mean():.1f}M")
print(f"  Maximum Loss: ${final_df['Loss_Millions'].max():.1f}M")
print(f"  Minimum Loss: ${final_df['Loss_Millions'].min():.1f}M")

# Show sample rows
print(f"\nSAMPLE OF FINAL DATASET (first 3 rows):")
print(final_df.head(3).to_string())


FINAL DATASET CHARACTERISTICS
Total movies in final dataset: 138
Dataset shape: (138, 10)

COLUMNS:
  Title: 138 non-null (0.0% null)
  Year: 138 non-null (0.0% null)
  Director: 136 non-null (1.4% null)
  Language: 136 non-null (1.4% null)
  Budget_Millions: 138 non-null (0.0% null)
  Loss_Millions: 138 non-null (0.0% null)
  Loss_Category: 138 non-null (0.0% null)
  IMDb_Rating: 136 non-null (1.4% null)
  Metascore: 133 non-null (3.6% null)
  Match_Status: 138 non-null (0.0% null)

LOSS CATEGORY DISTRIBUTION:
  Severe: 85 movies (61.6%)
  Catastrophic: 45 movies (32.6%)
  Moderate: 8 movies (5.8%)

MATCH STATUS DISTRIBUTION:
  Verified: 136 movies (98.6%)
  Not Found: 2 movies (1.4%)

RATING STATISTICS:
  IMDb Rating - Mean: 5.82
  IMDb Rating - Min: 2.10
  IMDb Rating - Max: 8.00

FINANCIAL STATISTICS:
  Average Budget: $128.8M
  Average Loss: $91.9M
  Maximum Loss: $218.0M
  Minimum Loss: $10.8M

SAMPLE OF FINAL DATASET (first 3 rows):
              Title  Year        Director    